# TrackMyself — job hunt insights

Read-only analysis of your applications: which platforms answer, whether fit predicts replies,
how long companies take, which weekdays work, and a suggested daily target mix.

**Setup (Google Colab)**

1. In TrackMyself go to **Settings → Integrations → API tokens** and create a token with the
   **Notebook (read-only)** preset. Copy it.
2. In Colab open the 🔑 **Secrets** panel (left sidebar) and add:
   - `TM_TOKEN` — the token (`tm_pat_…`)
   - `TM_URL` — optional, defaults to `https://trackmyself.webarden.tech`
3. Give this notebook access to both secrets, then **Runtime → Run all**.

In local Jupyter, set the same names as environment variables instead.
Nothing here writes to your account.

In [ ]:
import os, json, urllib.request, urllib.parse
from datetime import datetime

def secret(name, default=None):
    try:
        from google.colab import userdata  # type: ignore
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    return os.environ.get(name, default)

TM_URL = (secret("TM_URL") or "https://trackmyself.webarden.tech").rstrip("/")
TM_TOKEN = secret("TM_TOKEN")
if not TM_TOKEN:
    from getpass import getpass
    TM_TOKEN = getpass("TrackMyself token (tm_pat_…): ")
assert TM_TOKEN and TM_TOKEN.startswith("tm_pat_"), "Set TM_TOKEN to a TrackMyself personal access token."

def api(path, **params):
    query = ("?" + urllib.parse.urlencode({k: v for k, v in params.items() if v is not None})) if params else ""
    req = urllib.request.Request(f"{TM_URL}/api/v1{path}{query}", headers={"Authorization": f"Bearer {TM_TOKEN}", "Accept": "application/json"})
    with urllib.request.urlopen(req, timeout=60) as res:
        return json.loads(res.read().decode("utf-8"))

print(api("/me"))

## Load your applications

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

DAYS = 180
data = api("/insights", days=DAYS)
df = pd.DataFrame(data["applications"])
if df.empty:
    print("No applications in the last", DAYS, "days yet.")
else:
    for col in ["createdAt", "appliedAt", "postedAt", "deadline"]:
        df[col] = pd.to_datetime(df[col], utc=True, errors="coerce")
    applied = df[df["appliedAt"].notna()].copy()
    print(f"{len(df)} applications, {len(applied)} submitted, {int(applied['responded'].sum())} with a response.")
    display(df.head())

## Funnel by platform

*Responded* counts any move past Submitted, including a rejection: a fast no tells you more than silence.

In [ ]:
if not df.empty and not applied.empty:
    funnel = applied.groupby("platform").agg(
        applied=("id", "count"),
        responded=("responded", "sum"),
        interviewed=("interviewed", "sum"),
        offers=("offer", "sum"),
    )
    funnel["response_rate"] = (funnel["responded"] / funnel["applied"]).round(2)
    funnel["interview_rate"] = (funnel["interviewed"] / funnel["applied"]).round(2)
    display(funnel.sort_values("applied", ascending=False))
    funnel[["response_rate", "interview_rate"]].plot.bar(figsize=(8, 4), title="Rates by platform")
    plt.ylabel("share of applications"); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

## Does fit predict a reply?

Fit is the word-overlap score from the job inbox (0–100). Only applications that came through Job hunt have one.

In [ ]:
if not df.empty:
    scored = applied[applied["fitScore"].notna()].copy() if not applied.empty else applied
    if scored.empty:
        print("No scored applications yet. Apply through the job inbox to build this up.")
    else:
        scored["fit_band"] = pd.cut(scored["fitScore"], bins=[0, 40, 55, 70, 85, 100], include_lowest=True)
        by_fit = scored.groupby("fit_band", observed=True).agg(applied=("id", "count"), response_rate=("responded", "mean"), interview_rate=("interviewed", "mean"))
        display(by_fit.round(2))
        by_fit[["response_rate", "interview_rate"]].plot.bar(figsize=(8, 4), title="Outcome by fit band")
        plt.ylabel("share"); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

## How long until companies answer?

In [ ]:
if not df.empty:
    waits = applied["daysToFirstResponse"].dropna() if not applied.empty else pd.Series(dtype=float)
    if waits.empty:
        print("No responses recorded yet.")
    else:
        print(f"Median {waits.median():.0f} days, 80% within {waits.quantile(0.8):.0f} days.")
        waits.plot.hist(bins=range(0, int(waits.max()) + 3), figsize=(8, 3), title="Days to first response")
        plt.xlabel("days"); plt.tight_layout(); plt.show()
        print("Suggestion: set follow-ups about", int(max(5, waits.quantile(0.6))), "days after applying.")

## Which weekday do you apply on?

In [ ]:
if not df.empty and not applied.empty:
    names = ["Sun", "Mon", "Tue", "Wed", "Thu", "Fri", "Sat"]
    wd = applied.groupby("appliedWeekday").agg(applied=("id", "count"), response_rate=("responded", "mean"))
    wd.index = [names[int(i)] for i in wd.index]
    display(wd.round(2))

## Job inbox funnel

What happened to the leads found in the period.

In [ ]:
lf = pd.Series(data.get("leadFunnel", {}), name="leads").sort_values(ascending=False)
display(lf.to_frame()) if not lf.empty else print("No leads yet.")

## Suggested daily target mix

Spreads your current total daily target across platforms in proportion to each one's interview rate,
smoothed so a platform with few applications is not over- or under-weighted. Copy the numbers into
**Settings → Job hunt → Daily targets** if they look right.

In [ ]:
TOTAL_PER_DAY = 18   # your current total target
PRIOR_N, PRIOR_RATE = 10, 0.08   # pull small samples towards a typical 8% interview rate

if not df.empty and not applied.empty:
    g = applied.groupby("platform").agg(n=("id", "count"), k=("interviewed", "sum"))
    g["smoothed"] = (g["k"] + PRIOR_RATE * PRIOR_N) / (g["n"] + PRIOR_N)
    g["share"] = g["smoothed"] / g["smoothed"].sum()
    g["suggested_per_day"] = (g["share"] * TOTAL_PER_DAY).round().astype(int)
    display(g.sort_values("suggested_per_day", ascending=False)[["n", "k", "smoothed", "suggested_per_day"]].round(3))

## Today's report

In [ ]:
try:
    report = api("/report")
    print(json.dumps(report, indent=2, ensure_ascii=False)[:4000])
except Exception as e:
    print("Report unavailable:", e)